In [9]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data_path = (
    # "/home/chenzihao/workspace/cc2cc_atom/validate/ccdft_cc-pVDZ_cycle1-2690293_g2.csv"
    # "/home/chenzihao/workspace/cc2cc_atom/validate/ccdft_cc-pVDZ_cycle1-2690293_model.csv"
    "/home/chenzihao/workspace/cc2cc_atom/validate/ccdft_Def2-TZVPD_cycle1-2690293_g2.csv"
)
basis_args = data_path.split("/")[-1].split("_")[1]
print(basis_args)

with open("../cc2cc/utils/g2.json") as f:
    json_data = json.load(f)

data = pd.read_csv(data_path)

data["name"] = data["name"].str.split(f"_{basis_args}").str[0]

data_atomic_energy_dft = []
data_atomic_energy_ai = []
data_atomic_dft_ele = []
data_atomic_scf_ele = []
data_atomic_dft_dip = []
data_atomic_scf_dip = []

for i_name in data["name"]:
    if i_name not in json_data["reaction-atomic-energy"]:
        continue
    systems_list = json_data["reaction-atomic-energy"][i_name]["systems"]
    stoichiometry_list = json_data["reaction-atomic-energy"][i_name]["stoichiometry"]

    atomic_energy_dft = 0
    atomic_energy_ai = 0
    for i in range(len(systems_list)):
        atomic_energy_dft += data[data["name"] == systems_list[i]][
            "error_dft_ene"
        ].values[0] * int(stoichiometry_list[i])
        atomic_energy_ai += data[data["name"] == systems_list[i]][
            "error_scf_ene"
        ].values[0] * int(stoichiometry_list[i])
    data_atomic_energy_dft.append(atomic_energy_dft)
    data_atomic_energy_ai.append(atomic_energy_ai)

    data_atomic_dft_ele.append(
        data.loc[data["name"] == i_name, "error_dft_ele"].values[0]
    )
    data_atomic_scf_ele.append(
        data.loc[data["name"] == i_name, "error_scf_ele"].values[0]
    )
    data_atomic_dft_dip.append(
        data.loc[data["name"] == i_name, "error_dft_dip"].values[0]
    )
    data_atomic_scf_dip.append(
        data.loc[data["name"] == i_name, "error_scf_dip"].values[0]
    )

print(np.mean(np.abs(data_atomic_energy_dft)))
print(np.mean(np.abs(data_atomic_energy_ai)))

sorted_indices = np.argsort(np.abs(data_atomic_energy_dft))[::-1][:10]
sorted_data_atomic_energy_dft = np.array(data["name"])[sorted_indices]
print("dft", sorted_data_atomic_energy_dft)
sorted_indices = np.argsort(np.abs(data_atomic_energy_ai))[::-1][:10]
sorted_data_atomic_energy_ai = np.array(data["name"])[sorted_indices]
print("ai", sorted_data_atomic_energy_ai)

print("dft_ele", np.mean(np.abs(data_atomic_dft_ele)))
print("scf_ele", np.mean(np.abs(data_atomic_scf_ele)))
print("dft_dip", np.mean(np.abs(data_atomic_dft_dip)))
print("scf_dip", np.mean(np.abs(data_atomic_scf_dip)))

Def2-TZVPD
18.50838163604846
523.2097889282996
dft ['hcnh' 'o2' 'hooh' 'hs' 'f' 'n2h4' 'h2co' 'p' 'hoo' 'acetic']
ai ['p2' 'alf' 'o3' 'oclo' 'formic' 'b2h6' 'ocs' 'alcl' 'alh' 'alh3']
dft_ele 0.16858671723833507
scf_ele 0.0
dft_dip 0.02072005520498071
scf_dip 0.0


In [10]:
113.29000915592827 - 45.1735655726606 - 39.77333092424059

28.343112659027092